In [2]:
## Bye Week data
import os
import sys
import pandas as pd
from src.utils.db_utils import get_connection, execute_query

def byeweeks():
    """Bye weeks """
    query = """
    select *
    from stats.byeweek
    where season = '2024'
    """
    return  execute_query(query)

bye_weekdf = byeweeks()

print(bye_weekdf)

Successfully connected to the database!
    id                       created_at teamid  season  bye_week
0    1 2025-07-28 15:02:50.693830+00:00    LAC    2024         5
1    2 2025-07-28 15:02:50.693830+00:00    DEN    2024        14
2    3 2025-07-28 15:02:50.693830+00:00    KAN    2024         6
3    4 2025-07-28 15:02:50.693830+00:00    HOU    2024        14
4    5 2025-07-28 15:02:50.693830+00:00    BAL    2024        14
5    6 2025-07-28 15:02:50.693830+00:00    PIT    2024         9
6    7 2025-07-28 15:02:50.693830+00:00    ATL    2024        12
7    8 2025-07-28 15:02:50.693830+00:00    NYG    2024        11
8    9 2025-07-28 15:02:50.693830+00:00    CAR    2024        11
9   10 2025-07-28 15:02:50.693830+00:00    ARI    2024        11
10  11 2025-07-28 15:02:50.693830+00:00    CHI    2024         7
11  12 2025-07-28 15:02:50.693830+00:00    DET    2024         5
12  13 2025-07-28 15:02:50.693830+00:00    CIN    2024        12
13  14 2025-07-28 15:02:50.693830+00:00    CLE    

In [2]:
## Teams Stat, and Game Summary Join
def get_teamstats_with_game_context(gamesummaryid=None, season=None):
    """Get teamstats with game context fields populated"""
    
    # Joins teamstats with gamesummary. Populates Weeks from/since bye, and days since last game.
    query = """
    SELECT 
        ts.season,
        ts.week,
        ts.gamesummaryid,
        ts.teamid,
        ts.hometeamid,
        ts.awayteamid,
        ts.total_yards,
        -- Game context fields based on home/away logic
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.days_since_last_game_hometeam
            WHEN ts.teamid = gs.awayteamid THEN gs.days_since_last_game_awayteam
            ELSE NULL
        END as days_since_last_game,
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.hometeam_weeks_from_bye
            WHEN ts.teamid = gs.awayteamid THEN gs.awayteam_weeks_from_bye
            ELSE NULL
        END as weeks_from_bye
    FROM stats.teamstats ts
    LEFT JOIN stats.gamesummary gs ON ts.gamesummaryid = gs.gamesummaryid
    """
    
    # Add WHERE conditions based on parameters
    where_conditions = []
    if gamesummaryid:
        where_conditions.append(f"ts.gamesummaryid = '{gamesummaryid}'")
    if season:
        where_conditions.append(f"ts.season = {season}")
    
    if where_conditions:
        query += " WHERE " + " AND ".join(where_conditions)
    
    query += " ORDER BY ts.gamesummaryid, ts.teamid"
    
    return execute_query(query)

df_test = get_teamstats_with_game_context(gamesummaryid='gs-202410DETHOU')
#print("Test game results:")
print(df_test)

Successfully connected to the database!
Error executing query: 'OptionEngine' object has no attribute 'execute'
None


In [3]:
## Game Weather data
import os
import sys
import pandas as pd
from src.utils.db_utils import get_connection, execute_query
from sqlalchemy import create_engine, text
print(get_connection.__code__.co_filename)

def gameweather():
    """ Get game weather data """
    query = """
    SELECT 
        gamesummaryid, season, week, hometeamid, awayteamid, 
        kickoff_temperature_f, kickoff_wind_speed_mph, kickoff_wind_direction_deg, kickoff_humidity_pct,kickoff_pressure_mb, kickoff_precipitation_in, kickoff_weather_description,
        game_total_precipitation_in, game_total_rain_in, game_total_snow_in, 
        game_avg_pressure_mb, game_avg_wind_speed_mph, game_avg_wind_direction_deg

    From stats.gameweather
    WHERE season = '2024'          
    """
    return execute_query(query)

game_weather_df = gameweather()
print(game_weather_df)


c:\nfl predictive model\model\nfl_model\src\utils\db_utils.py
Successfully connected to the database!
Error executing query: 'OptionEngine' object has no attribute 'execute'
None


In [1]:
## Current Season Utility

import os
import sys
import pandas as pd
notebook_dir = os.path.abspath(os.path.dirname(''))
project_root = os.path.dirname(notebook_dir)
sys.path.append(project_root)
from src.utils.db_utils import get_connection
from src.utils.schedule_utility import get_full_schedule, get_current_week, get_team_schedule ## Import this utility File

## Get the full schedule
all_schedule = get_full_schedule()
all_schedule

## Get the current week
current_week = get_current_week()
current_week

## get the schedule for a specific team
phi_schedule = get_team_schedule('PHI')
phi_schedule


Successfully connected to the database!
✅ Retrieved 272 total games
Successfully connected to the database!
⚠️  No games found for current week
Successfully connected to the database!
🏈 Retrieved 17 games for PHI


,week,day,date,hometeam,awayteam,home_away,opponent
0,1,Thu,2025-09-04,PHI,DAL,HOME,DAL
1,2,Sun,2025-09-14,KAN,PHI,AWAY,KAN
2,3,Sun,2025-09-21,PHI,LAR,HOME,LAR
3,4,Sun,2025-09-28,TAM,PHI,AWAY,TAM
4,5,Sun,2025-10-05,PHI,DEN,HOME,DEN
5,6,Thu,2025-10-09,NYG,PHI,AWAY,NYG
6,7,Sun,2025-10-19,MIN,PHI,AWAY,MIN
7,8,Sun,2025-10-26,PHI,NYG,HOME,NYG
8,10,Mon,2025-11-10,GNB,PHI,AWAY,GNB
9,11,Sun,2025-11-16,PHI,DET,HOME,DET


In [ ]:
## Current Season table

def currentseasnschedule():
    """Queries Current Season Schedule"""
    query = """
        SELECT 
            season, week, awayteamid, 
            hometeamid, hometeam_weeks_from_bye, days_since_last_game_hometeam, -- Byes and rest days
            awayteam_weeks_from_bye, days_since_last_game_awayteam,
            kickoff_weather_description -- Weather data
        FROM stats.currentseasonschedule
    """
    return execute_query(query)

current_season = currentseasnschedule ()
print(current_season)